In [6]:
import os
from pydub import AudioSegment,silence
import librosa


In [11]:
import os
from pydub import AudioSegment, silence, effects

def make_uniform_segment(segment, target_ms):
    """Đảm bảo đoạn audio có độ dài chính xác bằng target_ms."""
    if len(segment) > target_ms:
        # Nếu dài hơn, cắt đúng phần đầu
        return segment[:target_ms]
    else:
        # Nếu ngắn hơn, bù thêm im lặng vào cuối
        silence_gap = AudioSegment.silent(duration=target_ms - len(segment))
        return segment + silence_gap

def extract_speech_from_audio(audio_dir, output_dir, target_sec=10):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    target_ms = target_sec * 1000  # Đổi sang mili giây

    for audio_file in os.listdir(audio_dir):
        # Hỗ trợ cả mp3 và wav
        if not audio_file.lower().endswith((".mp3", ".wav")):
            continue
            
        audio_path = os.path.join(audio_dir, audio_file)
        # Tự động nhận diện định dạng file
        audio = AudioSegment.from_file(audio_path)
        
        # 1. Chuẩn hóa về Mono và Sample Rate trước khi xử lý
        audio = audio.set_channels(1).set_frame_rate(22050)

        # 2. Tách dựa trên khoảng lặng
        # keep_silence=200 giúp đoạn nói tự nhiên hơn, không bị ngắt cụt
        chunks = silence.split_on_silence(
            audio, min_silence_len=600, silence_thresh=-45, keep_silence=300
        )

        final_segments = []
        for chunk in chunks:
            # Nếu chunk quá dài (VD: 25s), chia nó thành các đoạn target_ms (10s)
            if len(chunk) > target_ms:
                for i in range(0, len(chunk), target_ms):
                    sub_chunk = chunk[i : i + target_ms]
                    # Đảm bảo sub_chunk cuối cùng cũng đủ target_ms
                    final_segments.append(make_uniform_segment(sub_chunk, target_ms))
            else:
                # Nếu chunk ngắn (dưới 10s), bù thêm im lặng
                # Chỉ lấy nếu chunk không quá "rác" (ví dụ trên 1 giây mới lấy)
                if len(chunk) > 1000:
                    final_segments.append(make_uniform_segment(chunk, target_ms))

        # 3. Xuất file (giới hạn 10 đoạn đầu tiên như yêu cầu cũ của bạn)
        for i, segment in enumerate(final_segments[:10]):
            # Chuẩn hóa âm lượng lần cuối để đồng nhất về độ to
            segment = effects.normalize(segment)
            
            output_filename = f"segment_{i}.wav"
            output_path = os.path.join(output_dir, output_filename)
            segment.export(output_path, format="wav")

# --- Thực thi ---
audio_dir="/home/minwell/Documents/python project/voice_clone/data/raw"
output_dir="/home/minwell/Documents/python project/voice_clone/data/processed"
extract_speech_from_audio(audio_dir, output_dir, target_sec=10)

ModuleNotFoundError: No module named 'preprocess'